# Same MLP, PyTorch Edition

A clean PyTorch implementation of a fraud-detection MLP. Run side-by-side with the Keras notebook to internalize the differences.

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)

df = pd.read_parquet(Path('..') / 'data' / 'transactions.parquet')
df = pd.get_dummies(df, columns=['device_type', 'country'], drop_first=True)
for c in ['email_risk', 'device_entropy']:
    df[c] = df[c].fillna(df[c].median())
y = df['is_fraud'].values.astype(np.float32)
X = df.drop(columns=['is_fraud']).values.astype(np.float32)
X_tr, X_va, y_tr, y_va = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)
scaler = StandardScaler().fit(X_tr)
X_tr, X_va = scaler.transform(X_tr).astype(np.float32), scaler.transform(X_va).astype(np.float32)

In [ ]:
class FraudMLP(nn.Module):
    def __init__(self, in_dim, hidden=(64, 32), dropout=0.2):
        super().__init__()
        layers, prev = [], in_dim
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers += [nn.Linear(prev, 1)]
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x)

model = FraudMLP(X_tr.shape[1])
opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
pos_weight = torch.tensor((y_tr == 0).sum() / (y_tr == 1).sum(), dtype=torch.float32)
loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

ds = TensorDataset(torch.from_numpy(X_tr), torch.from_numpy(y_tr).unsqueeze(1))
loader = DataLoader(ds, batch_size=256, shuffle=True)

In [ ]:
for ep in range(1, 11):
    model.train()
    for xb, yb in loader:
        opt.zero_grad()
        loss_fn(model(xb), yb).backward()
        opt.step()
    model.eval()
    with torch.no_grad():
        probs = torch.sigmoid(model(torch.from_numpy(X_va))).numpy().ravel()
    print(f"ep {ep:02d}  ROC {roc_auc_score(y_va, probs):.4f}  PR {average_precision_score(y_va, probs):.4f}")

### Idioms to internalize

- `model.train()` / `model.eval()` toggles dropout & batchnorm. **Forgetting this is a classic bug.**
- `opt.zero_grad()` before `.backward()`. PyTorch accumulates gradients by default.
- `BCEWithLogitsLoss` not `Sigmoid + BCELoss`.